#  STEP 1: YOLOv8 POSE MODEL TRAINING

This step trains a YOLOv8 pose estimation model to detect cows and predict 6 keypoints

The model learns to locate cow body landmarks for subsequent measurement extraction

In [ ]:
# Path setup: notebooks live in notebooks/, but every path below is relative to the repo root
import os
from pathlib import Path

_here = Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "README.md").exists() and (_cand / "notebooks").is_dir():
        os.chdir(_cand)
        break
print("Working directory:", Path.cwd())

In [1]:
import torch

# 1. Check if PyTorch can see the GPU
print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. Check how many GPUs are detected (should be 1 or more)
print(f"Device Count: {torch.cuda.device_count()}")

# 3. Print the CUDA version PyTorch is linked to
print(f"PyTorch CUDA Version: {torch.version.cuda}")

# 4. (Optional) Check the name of your GPU
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Count: 1
PyTorch CUDA Version: 12.4
GPU Name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [9]:
from ultralytics import YOLO
import os

# Configuration
DATA_YAML = "configs/data.yaml" # Path to your dataset YAML file
MODEL_NAME = "models/pretrained/yolov8n-pose.pt" # Pre-trained model
EPOCHS = 100
IMGSZ = 640
# --- MODIFIED: Use -1 for auto-batching to find max size for 4GB VRAM ---
BATCH = 16 
PROJECT = "runs/pose"
NAME = "yolov8_cow_pose_v1"

# Load pre-trained YOLOv8 pose model
model = YOLO(MODEL_NAME)

# Train the model
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    # --- ADDED: Explicitly set device to 0 (first GPU) ---
    device=0, 
    project=PROJECT,
    name=NAME,
    patience=10, # Early stopping patience
    save=True,
    exist_ok=True,
    workers=0  
)

print("✅ YOLOv8 Pose training completed!")
print(f"Results saved in: {PROJECT}/{NAME}")

Ultralytics 8.3.203  Python-3.11.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_cow_pose_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.


# STEP 2: DATA PREPARATION & KEYPOINT VISUALIZATION

This step loads the dataset, visualizes keypoints with skeleton connections and prepares the data for measurement extraction pipeline


In [10]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

In [15]:
# Configuration
IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels" 
EXCEL_PATH = "data/measurements.xlsx"
IMAGE_ID_COL = "image_id"
NUM_KPTS = 6
SKELETON = [(0,1), (2,3), (4,5)]

# Create output directory
os.makedirs("results/keypoint_visualizations", exist_ok=True)

In [16]:
def read_yolo_keypoints(label_path, img_width, img_height):
    """Read YOLO format keypoints and convert to pixel coordinates"""
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    all_keypoints = []
    for line in lines:
        parts = list(map(float, line.strip().split()))
        if len(parts) < 5 + NUM_KPTS * 2:
            continue
            
        # Extract keypoints (skip class_id and bbox: first 5 values)
        kpt_data = parts[5:5 + NUM_KPTS * 2]
        keypoints = []
        
        for i in range(NUM_KPTS):
            x = kpt_data[i * 2] * img_width
            y = kpt_data[i * 2 + 1] * img_height
            keypoints.append([x, y])
            
        all_keypoints.append(np.array(keypoints))
    
    return all_keypoints

def draw_keypoints_skeleton(image, keypoints, skeleton_pairs):
    """Draw keypoints and skeleton on image"""
    img_copy = image.copy()
    
    # Draw skeleton lines
    for i, j in skeleton_pairs:
        if i < len(keypoints) and j < len(keypoints):
            pt1 = (int(keypoints[i][0]), int(keypoints[i][1]))
            pt2 = (int(keypoints[j][0]), int(keypoints[j][1]))
            cv2.line(img_copy, pt1, pt2, (0, 255, 0), 3)
    
    # Draw keypoints
    for idx, (x, y) in enumerate(keypoints):
        center = (int(x), int(y))
        cv2.circle(img_copy, center, 6, (0, 0, 255), -1)
        cv2.putText(img_copy, str(idx), (center[0] + 10, center[1]), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    return img_copy


In [18]:
# Load Excel measurements
df_measurements = pd.read_excel(EXCEL_PATH)
df_measurements[IMAGE_ID_COL] = df_measurements[IMAGE_ID_COL].astype(str)

In [20]:
# Visualize sample images with keypoints
sample_images = df_measurements[IMAGE_ID_COL].head(6).tolist()
plt.figure(figsize=(15, 10))
for idx, img_id in enumerate(sample_images):
    # Find image file
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = os.path.join(IMAGES_DIR, f"{img_id}{ext}")
        if os.path.exists(candidate):
            img_path = candidate
            break
    
    if not img_path:
        continue
        
    # Load image and keypoints
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    label_path = os.path.join(LABELS_DIR, f"{img_id}.txt")
    if os.path.exists(label_path):
        keypoints_list = read_yolo_keypoints(label_path, w, h)
        if keypoints_list:
            # Take first detection
            keypoints = keypoints_list[0]
            vis_img = draw_keypoints_skeleton(img, keypoints, SKELETON)
            
            # Plot
            plt.subplot(2, 3, idx + 1)
            plt.imshow(vis_img)
            plt.title(f"Image {img_id} - Keypoints")
            plt.axis('off')
            
            # Save visualization
            cv2.imwrite(f"results/keypoint_visualizations/{img_id}_keypoints.jpg", 
                       cv2.cvtColor(vis_img, cv2.COLOR_RGB2BGR))

plt.tight_layout()
plt.savefig("results/figures/keypoints_visualization_samples.jpg", dpi=150, bbox_inches='tight')
plt.show()

print("✅ Data preparation and visualization completed!")
print(f"Sample visualizations saved in 'results/keypoint_visualizations/' folder")

<Figure size 1500x1000 with 6 Axes>

✅ Data preparation and visualization completed!
Sample visualizations saved in 'visualizations/' folder


# STEP 3: FEATURE ENGINEERING & MEASUREMENT SCALING

This step computes all pairwise keypoint distances, establishes pixel-to-cm scaling using reference measurements, and creates feature table for ML training

In [21]:
import os
import cv2
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

In [22]:
# Configuration
IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"
EXCEL_PATH = "data/measurements.xlsx"
OUTPUT_CSV = "results/cow_features.csv"
IMAGE_ID_COL = "image_id"
NUM_KPTS = 6
TARGET_COLS = ["oblique_body_length_cm", "withers_height_cm", "girth_length_cm"]

In [23]:
# Reference keypoint pairs for each measurement (adjust based on your anatomy)
REF_PAIRS = {
    "oblique_body_length_cm": (0, 1),
    "withers_height_cm": (2, 3), 
    "girth_length_cm": (4, 5)
}

In [24]:
def load_image_keypoints(image_id):
    """Load image and corresponding keypoints"""
    # Find image file
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png']:
        candidate = os.path.join(IMAGES_DIR, f"{image_id}{ext}")
        if os.path.exists(candidate):
            img_path = candidate
            break
    
    if not img_path:
        return None, None
        
    # Load image
    img = cv2.imread(img_path)
    if img is None:
        return None, None
        
    h, w = img.shape[:2]
    
    # Load keypoints
    label_path = os.path.join(LABELS_DIR, f"{image_id}.txt")
    if not os.path.exists(label_path):
        return None, None
        
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    if not lines:
        return None, None
        
    # Parse first object's keypoints
    parts = list(map(float, lines[0].strip().split()))
    if len(parts) < 5 + NUM_KPTS * 2:
        return None, None
        
    kpt_data = parts[5:5 + NUM_KPTS * 2]
    keypoints = []
    
    for i in range(NUM_KPTS):
        x = kpt_data[i * 2] * w
        y = kpt_data[i * 2 + 1] * h
        keypoints.append([x, y])
    
    return np.array(keypoints), img

def compute_pairwise_distances(keypoints):
    """Compute all pairwise Euclidean distances between keypoints"""
    distances = {}
    for i, j in combinations(range(len(keypoints)), 2):
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        distances[f"{i}_{j}"] = dist
    return distances

In [25]:
# Load measurements data
df_meas = pd.read_excel(EXCEL_PATH)
df_meas[IMAGE_ID_COL] = df_meas[IMAGE_ID_COL].astype(str)

# Build feature dataset
records = []
scaling_data = []

In [28]:
print("Processing images and computing features...")
for idx, row in df_meas.iterrows():
    image_id = row[IMAGE_ID_COL]
    
    keypoints, img = load_image_keypoints(image_id)
    if keypoints is None:
        print(f"⚠️ Skipping image {image_id} - keypoints not found")
        continue
    
    # Compute all pairwise distances
    pairwise_dists = compute_pairwise_distances(keypoints)
    
    # Create feature record
    feature_record = {"image_id": image_id}
    feature_record.update(pairwise_dists)
    
    # Add ground truth measurements
    for col in TARGET_COLS:
        feature_record[col] = row[col]
    
    records.append(feature_record)
    
    # Collect data for scaling (using oblique length as reference)
    ref_pair = REF_PAIRS["oblique_body_length_cm"]
    ref_pixel_dist = np.linalg.norm(keypoints[ref_pair[0]] - keypoints[ref_pair[1]])
    true_length_cm = row["oblique_body_length_cm"]
    
    scaling_data.append({
        "image_id": image_id,
        "pixel_distance": ref_pixel_dist,
        "true_cm": true_length_cm
    })
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(df_meas)} images")
        
# Create features dataframe
features_df = pd.DataFrame(records)
print(f"✅ Feature table created with {len(features_df)} samples and {len(features_df.columns)-4} features")


Processing images and computing features...
Processed 10/138 images
Processed 20/138 images
Processed 30/138 images
Processed 40/138 images
Processed 50/138 images
Processed 60/138 images
Processed 70/138 images
Processed 80/138 images
Processed 90/138 images
Processed 100/138 images
Processed 110/138 images
Processed 120/138 images
Processed 130/138 images
✅ Feature table created with 276 samples and 15 features


# PIXEL TO CM SCALING

In [29]:
scaling_df = pd.DataFrame(scaling_data)

# Fit linear regression for pixel to cm conversion
X_scale = scaling_df["pixel_distance"].values.reshape(-1, 1)
y_scale = scaling_df["true_cm"].values

scaler = LinearRegression()
scaler.fit(X_scale, y_scale)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [30]:
# Plot scaling relationship
plt.figure(figsize=(10, 6))
plt.scatter(scaling_df["pixel_distance"], scaling_df["true_cm"], alpha=0.7)
plt.plot(X_scale, scaler.predict(X_scale), 'r-', linewidth=2)
plt.xlabel("Pixel Distance")
plt.ylabel("True Measurement (cm)")
plt.title(f"Pixel to CM Scaling\ncm = {scaler.coef_[0]:.3f} * pixels + {scaler.intercept_:.3f}")
plt.grid(True, alpha=0.3)
plt.savefig("results/figures/pixel_to_cm_scaling_plot.jpg", dpi=150, bbox_inches='tight')
plt.show()

print(f"📏 Scaling equation: cm = {scaler.coef_[0]:.4f} * pixels + {scaler.intercept_:.4f}")
print(f"📊 Scaling R² score: {scaler.score(X_scale, y_scale):.4f}")


<Figure size 1000x600 with 1 Axes>

📏 Scaling equation: cm = 0.0501 * pixels + 34.7894
📊 Scaling R² score: 0.3909


In [31]:
# Convert all pixel distances to cm using the scaler
features_cm_df = features_df.copy()
for col in features_cm_df.columns:
    if col not in ["image_id"] + TARGET_COLS:
        pixel_dists = features_cm_df[col].values.reshape(-1, 1)
        cm_dists = scaler.predict(pixel_dists)
        features_cm_df[col] = cm_dists

# Save features
features_cm_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Features saved to {OUTPUT_CSV}")
print(f"📋 Dataset shape: {features_cm_df.shape}")
print(f"🎯 Target variables: {TARGET_COLS}")

✅ Features saved to cow_features.csv
📋 Dataset shape: (276, 19)
🎯 Target variables: ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']


# STEP 4: MACHINE LEARNING MODEL TRAINING & EVALUATION

This step trains multiple regression models to predict body measurements from keypoint distances

Evaluates model performance and selects the best model for each measurement type


In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')

In [33]:
# Configuration
FEATURES_CSV = "results/cow_features.csv"
TARGET_COLS = ["oblique_body_length_cm", "withers_height_cm", "girth_length_cm"]
TEST_SIZE = 0.2
RANDOM_STATE = 42
MODELS_SAVE_DIR = "models/regressors"
os.makedirs(MODELS_SAVE_DIR, exist_ok=True)

In [34]:
# Load features data
print("📊 Loading features data...")
df = pd.read_csv(FEATURES_CSV)
print(f"Dataset shape: {df.shape}")
print(f"Features: {len(df.columns) - 4} | Samples: {len(df)}")

📊 Loading features data...
Dataset shape: (276, 19)
Features: 15 | Samples: 276


In [35]:
# Prepare features and targets
feature_cols = [col for col in df.columns if col not in ['image_id'] + TARGET_COLS]
X = df[feature_cols]
y = df[TARGET_COLS]

print(f"Feature columns: {feature_cols}")
print(f"Target distributions:")
for target in TARGET_COLS:
    print(f"  {target}: min={df[target].min():.1f}, max={df[target].max():.1f}, mean={df[target].mean():.1f}")

Feature columns: ['0_1', '0_2', '0_3', '0_4', '0_5', '1_2', '1_3', '1_4', '1_5', '2_3', '2_4', '2_5', '3_4', '3_5', '4_5']
Target distributions:
  oblique_body_length_cm: min=67.0, max=174.0, mean=139.5
  withers_height_cm: min=93.0, max=165.0, mean=122.6
  girth_length_cm: min=98.0, max=230.0, mean=183.7


In [36]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
)

print(f"\n📈 Train set: {X_train.shape[0]} samples")
print(f"📊 Test set: {X_test.shape[0]} samples")


📈 Train set: 220 samples
📊 Test set: 56 samples


In [37]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
joblib.dump(scaler, os.path.join(MODELS_SAVE_DIR, "feature_scaler.pkl"))

['trained_models\\feature_scaler.pkl']

MODEL DEFINITIONS

In [38]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso Regression": Lasso(alpha=0.1, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, verbosity=0),
    "Support Vector Regression": SVR(kernel='rbf', C=1.0)
}

MODEL TRAINING & EVALUATION

In [39]:
results = {}
feature_importance_data = {}

print("\n🚀 Training models...")
for model_name, model in models.items():
    print(f"\n📋 Training {model_name}...")
    model_results = {}
    
    # Train and evaluate for each target
    for i, target in enumerate(TARGET_COLS):
        # Train model
        if model_name in ["Linear Regression", "Ridge Regression", "Lasso Regression", "Support Vector Regression"]:
            model.fit(X_train_scaled, y_train[target])
            y_pred = model.predict(X_test_scaled)
        else:
            model.fit(X_train, y_train[target])
            y_pred = model.predict(X_test)
        
        # Calculate metrics
        mae = mean_absolute_error(y_test[target], y_pred)
        rmse = np.sqrt(mean_squared_error(y_test[target], y_pred))
        r2 = r2_score(y_test[target], y_pred)
        
        # Cross-validation score
        cv_scores = cross_val_score(model, X_train_scaled if model_name in ["Linear Regression", "Ridge", "Lasso", "SVR"] else X_train, 
                                  y_train[target], cv=5, scoring='r2')
        cv_r2_mean = cv_scores.mean()
        
        model_results[target] = {
            'MAE': mae,
            'RMSE': rmse, 
            'R2': r2,
            'CV_R2_Mean': cv_r2_mean,
            'Predictions': y_pred
        }
        
        print(f"  {target}: MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.3f}, CV-R²={cv_r2_mean:.3f}")
    
    results[model_name] = model_results
    
    # Save model
    if model_name in ["Random Forest", "Gradient Boosting", "XGBoost"]:
        joblib.dump(model, os.path.join(MODELS_SAVE_DIR, f"{model_name.replace(' ', '_').lower()}_model.pkl"))
    else:
        joblib.dump(model, os.path.join(MODELS_SAVE_DIR, f"{model_name.replace(' ', '_').lower()}_model.pkl"))



🚀 Training models...

📋 Training Linear Regression...
  oblique_body_length_cm: MAE=10.82, RMSE=13.10, R²=0.734, CV-R²=0.576
  withers_height_cm: MAE=10.76, RMSE=13.40, R²=0.150, CV-R²=0.168
  girth_length_cm: MAE=14.49, RMSE=17.93, R²=0.564, CV-R²=0.460

📋 Training Ridge Regression...
  oblique_body_length_cm: MAE=12.10, RMSE=14.67, R²=0.666, CV-R²=0.570
  withers_height_cm: MAE=11.45, RMSE=13.74, R²=0.106, CV-R²=0.174
  girth_length_cm: MAE=17.34, RMSE=19.71, R²=0.474, CV-R²=0.460

📋 Training Lasso Regression...
  oblique_body_length_cm: MAE=12.26, RMSE=14.83, R²=0.658, CV-R²=0.525
  withers_height_cm: MAE=11.26, RMSE=13.71, R²=0.109, CV-R²=0.184
  girth_length_cm: MAE=17.34, RMSE=19.74, R²=0.472, CV-R²=0.404

📋 Training Random Forest...
  oblique_body_length_cm: MAE=2.63, RMSE=4.40, R²=0.970, CV-R²=0.912
  withers_height_cm: MAE=1.34, RMSE=2.26, R²=0.976, CV-R²=0.929
  girth_length_cm: MAE=1.85, RMSE=2.88, R²=0.989, CV-R²=0.942

📋 Training Gradient Boosting...
  oblique_body_length

# RESULTS ANALYSIS

In [40]:
print("\n" + "="*60)
print("📊 FINAL MODEL COMPARISON")
print("="*60)

# Create results summary
summary_data = []
best_models = {}

for target in TARGET_COLS:
    print(f"\n🎯 {target}:")
    print("-" * 40)
    
    target_results = []
    for model_name in models.keys():
        r2 = results[model_name][target]['R2']
        mae = results[model_name][target]['MAE']
        target_results.append((model_name, r2, mae))
        
        summary_data.append({
            'Target': target,
            'Model': model_name,
            'R2': r2,
            'MAE': mae,
            'RMSE': results[model_name][target]['RMSE'],
            'CV_R2_Mean': results[model_name][target]['CV_R2_Mean']
        })
    
    # Sort by R2 score and select best model
    target_results.sort(key=lambda x: x[1], reverse=True)
    best_model_name, best_r2, best_mae = target_results[0]
    best_models[target] = best_model_name
    
    print(f"🏆 BEST: {best_model_name} (R²: {best_r2:.3f}, MAE: {best_mae:.2f} cm)")
    
    # Print all models for this target
    for model_name, r2, mae in target_results:
        print(f"  {model_name:20} R²: {r2:.3f} | MAE: {mae:.2f} cm")

# Save results summary
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv("results/model_performance_summary.csv", index=False)
print(f"\n✅ Model performance summary saved to 'results/model_performance_summary.csv'")


📊 FINAL MODEL COMPARISON

🎯 oblique_body_length_cm:
----------------------------------------
🏆 BEST: XGBoost (R²: 0.988, MAE: 1.07 cm)
  XGBoost              R²: 0.988 | MAE: 1.07 cm
  Gradient Boosting    R²: 0.987 | MAE: 1.70 cm
  Random Forest        R²: 0.970 | MAE: 2.63 cm
  Linear Regression    R²: 0.734 | MAE: 10.82 cm
  Ridge Regression     R²: 0.666 | MAE: 12.10 cm
  Lasso Regression     R²: 0.658 | MAE: 12.26 cm
  Support Vector Regression R²: 0.426 | MAE: 13.29 cm

🎯 withers_height_cm:
----------------------------------------
🏆 BEST: XGBoost (R²: 0.991, MAE: 0.47 cm)
  XGBoost              R²: 0.991 | MAE: 0.47 cm
  Gradient Boosting    R²: 0.983 | MAE: 1.22 cm
  Random Forest        R²: 0.976 | MAE: 1.34 cm
  Support Vector Regression R²: 0.173 | MAE: 8.46 cm
  Linear Regression    R²: 0.150 | MAE: 10.76 cm
  Lasso Regression     R²: 0.109 | MAE: 11.26 cm
  Ridge Regression     R²: 0.106 | MAE: 11.45 cm

🎯 girth_length_cm:
----------------------------------------
🏆 BEST: X

# VISUALIZATION

In [41]:
print("\n📈 Generating performance visualizations...")

# 1. Model comparison plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, target in enumerate(TARGET_COLS):
    model_names = list(models.keys())
    r2_scores = [results[model][target]['R2'] for model in model_names]
    mae_scores = [results[model][target]['MAE'] for model in model_names]
    
    x = np.arange(len(model_names))
    width = 0.35
    
    axes[idx].bar(x - width/2, r2_scores, width, label='R² Score', alpha=0.8)
    axes[idx].bar(x + width/2, mae_scores, width, label='MAE (cm)', alpha=0.8)
    axes[idx].set_xlabel('Models')
    axes[idx].set_title(f'{target} Prediction Performance')
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(model_names, rotation=45, ha='right')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/model_performance_comparison.jpg', dpi=150, bbox_inches='tight')
plt.show()


📈 Generating performance visualizations...


<Figure size 1800x600 with 3 Axes>

In [42]:
# 2. Best model predictions vs actual
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for idx, target in enumerate(TARGET_COLS):
    best_model_name = best_models[target]
    y_pred = results[best_model_name][target]['Predictions']
    y_true = y_test[target].values
    
    axes[idx].scatter(y_true, y_pred, alpha=0.7)
    axes[idx].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[idx].set_xlabel('Actual (cm)')
    axes[idx].set_ylabel('Predicted (cm)')
    axes[idx].set_title(f'{target}\n{best_model_name}\nR²: {results[best_model_name][target]["R2"]:.3f}')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/best_models_predictions.jpg', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Model training completed!")
print(f"💾 Models saved in '{MODELS_SAVE_DIR}/' directory")
print(f"🏆 Best models selected:")
for target, model in best_models.items():
    print(f"   {target}: {model}")

<Figure size 1800x600 with 3 Axes>


✅ Model training completed!
💾 Models saved in 'trained_models/' directory
🏆 Best models selected:
   oblique_body_length_cm: XGBoost
   withers_height_cm: XGBoost
   girth_length_cm: XGBoost


================================================================================
# STEP 5: FINAL INFERENCE PIPELINE

This step combines YOLO pose detection with trained ML models to create an end-to-end system that takes new cow images and predicts body measurements

In [43]:
import cv2
import numpy as np
import pandas as pd
import joblib
from ultralytics import YOLO
from itertools import combinations
import matplotlib.pyplot as plt

In [48]:
# CONFIGURATION

# Paths
YOLO_MODEL_PATH = "runs/pose/yolov8_cow_pose_v1/weights/best.pt"  # Update with your trained YOLO path
FEATURE_SCALER_PATH = "models/regressors/feature_scaler.pkl"
ML_MODELS_DIR = "models/regressors"

# Best models from previous step (update based on your results)
BEST_MODELS = {
    "oblique_body_length_cm": "xgboost_model.pkl",
    "withers_height_cm": "xgboost_model.pkl", 
    "girth_length_cm": "xgboost_model.pkl"
}

# Keypoint configuration (must match training)
NUM_KPTS = 6
SKELETON_PAIRS = [(0, 1), (2, 3), (4, 5)]

# Pixel to CM scaling (from Step 3)
PIXEL_TO_CM_SLOPE = 0.0501
PIXEL_TO_CM_INTERCEPT = 34.7894


In [49]:
# LOAD MODELS

print("🔄 Loading trained models...")

# Load YOLO pose model
yolo_model = YOLO(YOLO_MODEL_PATH)
print("✅ YOLO pose model loaded")

# Load feature scaler
feature_scaler = joblib.load(FEATURE_SCALER_PATH)
print("✅ Feature scaler loaded")

# Load ML models
ml_models = {}
for target, model_file in BEST_MODELS.items():
    model_path = f"{ML_MODELS_DIR}/{model_file}"
    ml_models[target] = joblib.load(model_path)
    print(f"✅ {target} model loaded: {model_file}")

🔄 Loading trained models...
✅ YOLO pose model loaded
✅ Feature scaler loaded
✅ oblique_body_length_cm model loaded: xgboost_model.pkl
✅ withers_height_cm model loaded: xgboost_model.pkl
✅ girth_length_cm model loaded: xgboost_model.pkl


In [50]:
# HELPER FUNCTIONS

def extract_keypoints_from_yolo(results):
    """Extract keypoints from YOLO results"""
    if len(results) == 0 or len(results[0].keypoints.xy) == 0:
        return None
    
    # Get keypoints from first detected object
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    return keypoints

def compute_all_pairwise_distances(keypoints):
    """Compute all pairwise Euclidean distances between keypoints"""
    distances = []
    for i, j in combinations(range(len(keypoints)), 2):
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        distances.append(dist)
    return np.array(distances)

def pixel_to_cm_conversion(pixel_distances):
    """Convert pixel distances to cm using trained scaling"""
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    return cm_distances

def draw_comprehensive_visualization(image, keypoints, measurements, output_path):
    """Draw keypoints, skeleton, and measurements on image"""
    img_vis = image.copy()
    
    # Draw skeleton
    for i, j in SKELETON_PAIRS:
        if i < len(keypoints) and j < len(keypoints):
            pt1 = (int(keypoints[i][0]), int(keypoints[i][1]))
            pt2 = (int(keypoints[j][0]), int(keypoints[j][1]))
            cv2.line(img_vis, pt1, pt2, (0, 255, 0), 3)
    
    # Draw keypoints
    for idx, (x, y) in enumerate(keypoints):
        center = (int(x), int(y))
        cv2.circle(img_vis, center, 6, (0, 0, 255), -1)
        cv2.putText(img_vis, str(idx), (center[0] + 10, center[1]), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    # Add measurements text
    y_offset = 40
    for measurement, value in measurements.items():
        text = f"{measurement}: {value:.1f} cm"
        cv2.putText(img_vis, text, (20, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        y_offset += 35
    
    # Save visualization
    cv2.imwrite(output_path, img_vis)
    return img_vis

In [52]:
# HELPER FUNCTIONS

def extract_keypoints_from_yolo(results):
    """Extract keypoints from YOLO results"""
    if len(results) == 0 or len(results[0].keypoints.xy) == 0:
        return None
    
    # Get keypoints from first detected object
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    return keypoints

def compute_all_pairwise_distances(keypoints):
    """Compute all pairwise Euclidean distances between keypoints"""
    distances = []
    for i, j in combinations(range(len(keypoints)), 2):
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        distances.append(dist)
    return np.array(distances)

def pixel_to_cm_conversion(pixel_distances):
    """Convert pixel distances to cm using trained scaling"""
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    return cm_distances

def draw_comprehensive_visualization(image, keypoints, measurements, output_path):
    """Draw keypoints, skeleton, and measurements on image"""
    img_vis = image.copy()
    
    # Draw skeleton
    for i, j in SKELETON_PAIRS:
        if i < len(keypoints) and j < len(keypoints):
            pt1 = (int(keypoints[i][0]), int(keypoints[i][1]))
            pt2 = (int(keypoints[j][0]), int(keypoints[j][1]))
            cv2.line(img_vis, pt1, pt2, (0, 255, 0), 3)
    
    # Draw keypoints
    for idx, (x, y) in enumerate(keypoints):
        center = (int(x), int(y))
        cv2.circle(img_vis, center, 6, (0, 0, 255), -1)
        cv2.putText(img_vis, str(idx), (center[0] + 10, center[1]), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    # Add measurements text
    y_offset = 40
    for measurement, value in measurements.items():
        text = f"{measurement}: {value:.1f} cm"
        cv2.putText(img_vis, text, (20, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        y_offset += 35
    
    # Save visualization
    cv2.imwrite(output_path, img_vis)
    return img_vis

def predict_measurements_from_image(image_path, save_visualization=True):
    """Main function to predict measurements from a single image"""
    print(f"\n🔍 Processing image: {image_path}")
    

    # STEP 1: YOLO POSE DETECTION

    results = yolo_model.predict(image_path, imgsz=640, conf=0.5)
    
    # Extract keypoints
    keypoints = extract_keypoints_from_yolo(results)
    if keypoints is None:
        print("❌ No keypoints detected in the image")
        return None, None
    
    print(f"✅ Detected {len(keypoints)} keypoints")
    

    # STEP 2: FEATURE EXTRACTION

    pixel_distances = compute_all_pairwise_distances(keypoints)
    print(f"✅ Computed {len(pixel_distances)} pairwise distances")
    
    # Convert to cm
    cm_distances = pixel_to_cm_conversion(pixel_distances)
    
    # Ensure correct feature dimension (should be 15 for 6 keypoints: C(6,2)=15)
    expected_features = 15  # 6 choose 2 = 15 pairwise combinations
    if len(cm_distances) != expected_features:
        print(f"⚠️ Warning: Expected {expected_features} features, got {len(cm_distances)}")
        # Pad or truncate to match training
        if len(cm_distances) < expected_features:
            padded = np.zeros(expected_features)
            padded[:len(cm_distances)] = cm_distances
            cm_distances = padded
        else:
            cm_distances = cm_distances[:expected_features]
    
    # Scale features (same as training)
    features_scaled = feature_scaler.transform([cm_distances])
    

    # STEP 3: ML PREDICTION

    measurements = {}
    for target, model in ml_models.items():
        prediction = model.predict(features_scaled)[0]
        measurements[target] = prediction
    

    # STEP 4: VISUALIZATION

    if save_visualization:
        image = cv2.imread(image_path)
        output_viz_path = image_path.replace('.jpg', '_measurements.jpg').replace('.png', '_measurements.jpg')
        vis_image = draw_comprehensive_visualization(image, keypoints, measurements, output_viz_path)
        print(f"✅ Visualization saved: {output_viz_path}")
    
    return measurements, keypoints


# BATCH PROCESSING FUNCTION

def process_multiple_images(image_paths, output_csv="batch_predictions.csv"):
    """Process multiple images and save results to CSV"""
    all_results = []
    
    for image_path in image_paths:
        try:
            measurements, keypoints = predict_measurements_from_image(image_path, save_visualization=True)
            if measurements:
                result = {"image_path": image_path}
                result.update(measurements)
                all_results.append(result)
                print(f"✅ Completed: {image_path}")
            else:
                print(f"❌ Failed: {image_path}")
        except Exception as e:
            print(f"❌ Error processing {image_path}: {str(e)}")
    
    # Save batch results
    if all_results:
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(output_csv, index=False)
        print(f"\n✅ Batch results saved to: {output_csv}")
        print(f"📊 Processed {len(all_results)} images successfully")
    
    return all_results


In [55]:
if __name__ == "__main__":
    print("🐄 VETRA: Vision-Enabled Type Recording Assistant")
    print("=" * 50)
    
    # Example 1: Single image processing
    test_image = "data/images/29.png"  # Replace with your test image path
    
    try:
        measurements, keypoints = predict_measurements_from_image(test_image)
        
        if measurements:
            print("\n📏 PREDICTED MEASUREMENTS:")
            print("=" * 30)
            for measurement, value in measurements.items():
                print(f"{measurement:25}: {value:6.1f} cm")
            
            # Display visualization
            result_image = cv2.imread(test_image.replace('.jpg', '_measurements.jpg').replace('.png', '_measurements.jpg'))
            plt.figure(figsize=(12, 8))
            plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
            plt.title("Cow Body Measurements Prediction")
            plt.axis('off')
            plt.show()
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
    
    # Example 2: Batch processing (uncomment to use)
    """
    image_folder = "test_images/"
    image_paths = [os.path.join(image_folder, f) for f in os.listdir(image_folder) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if image_paths:
        batch_results = process_multiple_images(image_paths)
    else:
        print("❌ No images found in test_images folder")
    """

🐄 VETRA: Vision-Enabled Type Recording Assistant

🔍 Processing image: data/images/29.png

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 38.3ms
Speed: 12.0ms preprocess, 38.3ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
✅ Detected 6 keypoints
✅ Computed 15 pairwise distances
✅ Visualization saved: data/images/29_measurements.jpg

📏 PREDICTED MEASUREMENTS:
oblique_body_length_cm   :   99.2 cm
withers_height_cm        :   99.2 cm
girth_length_cm          :   99.2 cm


<Figure size 1200x800 with 1 Axes>

In [57]:
# ==============================
# STEP 6: DIAGNOSTIC TOOL - FINDING WHY PREDICTIONS ARE CONSTANT
# ==============================
# This step helps identify why the model gives constant predictions
# by examining each stage of the pipeline and comparing with training data

import numpy as np
import pandas as pd
import joblib
from ultralytics import YOLO
from itertools import combinations
import matplotlib.pyplot as plt

# Load the original training data for comparison
train_features = pd.read_csv("results/cow_features.csv")
print("📊 Training data statistics:")
print(f"Dataset shape: {train_features.shape}")
print(f"Features: {len(train_features.columns) - 4}")

# Display feature ranges from training data
feature_cols = [col for col in train_features.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
print(f"\n📈 Training feature ranges:")
for col in feature_cols[:5]:  # Show first 5 features
    min_val = train_features[col].min()
    max_val = train_features[col].max()
    mean_val = train_features[col].mean()
    print(f"  {col}: {min_val:.1f} to {max_val:.1f} (mean: {mean_val:.1f})")

def diagnostic_pipeline(image_path):
    """Run diagnostic pipeline to identify where constant predictions occur"""
    print(f"\n🔍 DIAGNOSTIC FOR: {image_path}")
    print("=" * 60)
    
    # Load models
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    feature_scaler = joblib.load("models/regressors/feature_scaler.pkl")
    ml_model = joblib.load("models/regressors/xgboost_model.pkl")
    
    # Step 1: YOLO Keypoint Detection
    print("\n1. 🎯 YOLO KEYPOINT DETECTION")
    results = yolo_model.predict(image_path, imgsz=640)
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    print(f"   Keypoints shape: {keypoints.shape}")
    print(f"   Keypoints values:")
    for i, kp in enumerate(keypoints):
        print(f"     KP{i}: ({kp[0]:.1f}, {kp[1]:.1f})")
    
    # Step 2: Pairwise Distance Calculation
    print("\n2. 📏 PAIRWISE DISTANCES (Pixels)")
    pixel_distances = []
    for i, j in combinations(range(len(keypoints)), 2):
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        pixel_distances.append(dist)
        print(f"     Distance {i}-{j}: {dist:.1f} pixels")
    
    pixel_distances = np.array(pixel_distances)
    
    # Step 3: Pixel to CM Conversion
    print("\n3. 🔄 PIXEL TO CM CONVERSION")
    PIXEL_TO_CM_SLOPE = 0.0501
    PIXEL_TO_CM_INTERCEPT = 34.7894
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    print(f"   CM distances range: {cm_distances.min():.1f} to {cm_distances.max():.1f} cm")
    print(f"   First 5 CM distances: {cm_distances[:5]}")
    
    # Step 4: Feature Scaling
    print("\n4. ⚖️ FEATURE SCALING")
    print(f"   Input features shape: {cm_distances.shape}")
    
    # Check if we have correct number of features
    expected_features = len(feature_cols)
    if len(cm_distances) != expected_features:
        print(f"   ⚠️ FEATURE COUNT MISMATCH!")
        print(f"   Expected: {expected_features}, Got: {len(cm_distances)}")
        # This is likely the problem!
    
    # Scale features
    features_scaled = feature_scaler.transform([cm_distances])
    print(f"   Scaled features range: {features_scaled.min():.3f} to {features_scaled.max():.3f}")
    print(f"   First 5 scaled features: {features_scaled[0][:5]}")
    
    # Step 5: Compare with training distribution
    print("\n5. 📊 COMPARISON WITH TRAINING DATA")
    train_feature_means = train_features[feature_cols].mean().values
    print(f"   Training feature means range: {train_feature_means.min():.1f} to {train_feature_means.max():.1f}")
    print(f"   Current features range: {cm_distances.min():.1f} to {cm_distances.max():.1f}")
    
    # Step 6: Model Prediction
    print("\n6. 🤖 MODEL PREDICTION")
    predictions = ml_model.predict(features_scaled)
    print(f"   Raw predictions: {predictions}")
    
    return keypoints, pixel_distances, cm_distances, features_scaled, predictions

# Run diagnostic on multiple test images
test_images = ["data/images/29.png", "data/images/30.png", "data/images/121.jpg", "data/images/132.jpg"]  # Add more test images

all_results = []
for img_path in test_images:
    try:
        kp, px_dists, cm_dists, features, preds = diagnostic_pipeline(img_path)
        all_results.append({
            'image': img_path,
            'keypoints_variation': np.std(kp),  # Measure how much keypoints vary
            'pixel_dists_range': np.ptp(px_dists),  # Range of pixel distances
            'cm_dists_range': np.ptp(cm_dists),     # Range of cm distances
            'predictions': preds[0] if len(preds) > 0 else 0
        })
    except Exception as e:
        print(f"❌ Error with {img_path}: {e}")

# Analyze results
if all_results:
    results_df = pd.DataFrame(all_results)
    print("\n" + "=" * 60)
    print("📈 DIAGNOSTIC SUMMARY")
    print("=" * 60)
    print(results_df.to_string(index=False))
    
    # Check for constant predictions
    if len(results_df['predictions'].unique()) == 1:
        print("\n🚨 PROBLEM IDENTIFIED: CONSTANT PREDICTIONS")
        print("Most likely causes:")
        print("1. 🔑 Keypoints not varying between images")
        print("2. 📐 Wrong number of features being fed to model")
        print("3. ⚖️ Feature scaling issue")
        print("4. 🎯 YOLO model detecting same keypoints regardless of input")

📊 Training data statistics:
Dataset shape: (276, 19)
Features: 15

📈 Training feature ranges:
  0_1: 117.1 to 168.9 (mean: 139.5)
  0_2: 62.7 to 96.1 (mean: 78.8)
  0_3: 76.8 to 103.0 (mean: 87.9)
  0_4: 81.5 to 117.8 (mean: 97.8)
  0_5: 78.5 to 103.6 (mean: 86.9)

🔍 DIAGNOSTIC FOR: data/images/29.png

1. 🎯 YOLO KEYPOINT DETECTION

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 39.1ms
Speed: 5.1ms preprocess, 39.1ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)
   Keypoints shape: (6, 2)
   Keypoints values:
     KP0: (1463.9, 1630.6)
     KP1: (3736.4, 864.2)
     KP2: (1929.9, 703.9)
     KP3: (1778.2, 2812.8)
     KP4: (2676.7, 737.0)
     KP5: (2675.2, 1809.4)

2. 📏 PAIRWISE DISTANCES (Pixels)
     Distance 0-1: 2398.3 pixels
     Distance 0-2: 1037.3 pixels
     Distance 0-3: 1223.2 pixels
     Distance 0-4: 1506.5 pixels
     Distance 0-5: 1224.4 pixels
     Distance 1-2: 1813.6 pixels
     Distance 1-3: 2762.5 pixels
     Dista

In [59]:
# ==============================
# STEP 7: CORRECTED INFERENCE PIPELINE
# ==============================
# Fixed version that ensures proper feature dimension matching

import cv2
import numpy as np
import joblib
from ultralytics import YOLO
from itertools import combinations

def fixed_predict_measurements(image_path):
    """Fixed version with proper feature handling"""
    print(f"\n🔍 Processing: {image_path}")
    
    # Load models
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    feature_scaler = joblib.load("models/regressors/feature_scaler.pkl")
    ml_models = {
        "oblique_body_length_cm": joblib.load("models/regressors/xgboost_model.pkl"),
        "withers_height_cm": joblib.load("models/regressors/xgboost_model.pkl"),
        "girth_length_cm": joblib.load("models/regressors/xgboost_model.pkl")
    }
    
    # Load training feature info to get correct dimension
    train_features = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in train_features.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    EXPECTED_FEATURES = len(feature_cols)
    print(f"📐 Expected features: {EXPECTED_FEATURES}")
    
    # YOLO detection
    results = yolo_model.predict(image_path, imgsz=640)
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    print(f"🎯 Detected {len(keypoints)} keypoints")
    
    # Compute pairwise distances - MUST MATCH TRAINING ORDER!
    pixel_distances = []
    for i, j in combinations(range(len(keypoints)), 2):
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        pixel_distances.append(dist)
    
    pixel_distances = np.array(pixel_distances)
    
    # Convert to cm
    PIXEL_TO_CM_SLOPE = 0.0501
    PIXEL_TO_CM_INTERCEPT = 34.7894
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    
    # CRITICAL: Ensure correct feature dimension
    if len(cm_distances) != EXPECTED_FEATURES:
        print(f"⚠️ Adjusting features: {len(cm_distances)} → {EXPECTED_FEATURES}")
        # This is likely your problem - feature count mismatch!
        if len(cm_distances) > EXPECTED_FEATURES:
            cm_distances = cm_distances[:EXPECTED_FEATURES]
        else:
            # Pad with mean values from training (better than zeros)
            padded = np.full(EXPECTED_FEATURES, train_features[feature_cols].mean().mean())
            padded[:len(cm_distances)] = cm_distances
            cm_distances = padded
    
    # Scale features
    features_scaled = feature_scaler.transform([cm_distances])
    
    # Predict
    measurements = {}
    for target, model in ml_models.items():
        prediction = model.predict(features_scaled)[0]
        measurements[target] = prediction
    
    print("📏 PREDICTED MEASUREMENTS:")
    for measurement, value in measurements.items():
        print(f"  {measurement}: {value:.1f} cm")
    
    return measurements

# Test the fixed pipeline
test_image = "data/images/29.png"
fixed_predict_measurements(test_image)


🔍 Processing: data/images/29.png
📐 Expected features: 15

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 39.0ms
Speed: 4.6ms preprocess, 39.0ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
🎯 Detected 6 keypoints
📏 PREDICTED MEASUREMENTS:
  oblique_body_length_cm: 99.2 cm
  withers_height_cm: 99.2 cm
  girth_length_cm: 99.2 cm


{'oblique_body_length_cm': np.float32(99.22779),
 'withers_height_cm': np.float32(99.22779),
 'girth_length_cm': np.float32(99.22779)}

In [61]:
# ==============================
# STEP 8: MODEL VALIDATION & RETRAINING
# ==============================
# This step validates the trained models and retrains them if necessary

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib
import matplotlib.pyplot as plt

# Load the features data
print("📊 Loading features data for model validation...")
df = pd.read_csv("results/cow_features.csv")
feature_cols = [col for col in df.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
target_cols = ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']

print(f"Dataset shape: {df.shape}")
print(f"Features: {len(feature_cols)}, Targets: {len(target_cols)}")

# Check target distributions
print("\n🎯 Target distributions:")
for target in target_cols:
    print(f"  {target}: min={df[target].min():.1f}, max={df[target].max():.1f}, mean={df[target].mean():.1f}")

# Test the existing model
def test_existing_model():
    """Test if the existing model works correctly"""
    print("\n🧪 TESTING EXISTING MODEL...")
    
    # Load the existing model
    try:
        existing_model = joblib.load("models/regressors/xgboost_model.pkl")
        print("✅ Existing model loaded")
        
        # Prepare data
        X = df[feature_cols]
        y = df[target_cols[0]]  # Test with first target
        
        # Make predictions on training data itself
        predictions = existing_model.predict(X)
        
        # Check if predictions are constant
        unique_predictions = np.unique(predictions)
        print(f"📈 Unique predictions: {len(unique_predictions)}")
        print(f"📊 Prediction range: {predictions.min():.2f} to {predictions.max():.2f}")
        
        if len(unique_predictions) <= 3:
            print("🚨 MODEL IS BROKEN - Outputting constant predictions!")
            return False
        else:
            print("✅ Model seems to be working")
            return True
            
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return False

# Test the model
model_works = test_existing_model()

if not model_works:
    print("\n🔄 RETRAINING MODELS...")
    
    # Split data
    X = df[feature_cols]
    
    # Train separate models for each target
    models = {}
    performance = {}
    
    for target in target_cols:
        print(f"\n🎯 Training model for: {target}")
        y = df[target]
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        # Train XGBoost model
        model = XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric='mae'
        )
        
        model.fit(X_train, y_train, 
                 eval_set=[(X_test, y_test)],
                 verbose=False)
        
        # Evaluate
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        models[target] = model
        performance[target] = {'MAE': mae, 'R2': r2}
        
        print(f"  ✅ Trained - MAE: {mae:.2f} cm, R²: {r2:.3f}")
        
        # Save model
        joblib.dump(model, f"models/regressors/{target}_model.pkl")
    
    # Print performance summary
    print("\n📊 RETRAINED MODELS PERFORMANCE:")
    for target, perf in performance.items():
        print(f"  {target}: MAE={perf['MAE']:.2f} cm, R²={perf['R2']:.3f}")
    
    # Test the retrained model
    print("\n🧪 TESTING RETRAINED MODEL...")
    test_model = models[target_cols[0]]
    test_predictions = test_model.predict(X.head(5))
    print(f"  Sample predictions: {test_predictions}")
    print(f"  Prediction range: {test_predictions.min():.2f} to {test_predictions.max():.2f}")

# ==============================
# CORRECTED INFERENCE PIPELINE WITH RETRAINED MODELS
# ==============================

def corrected_inference_pipeline(image_path):
    """Corrected pipeline with proper model handling"""
    from ultralytics import YOLO
    from itertools import combinations
    
    print(f"\n🔍 Processing: {image_path}")
    
    # Load YOLO model
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    
    # Load CORRECT models (one per target)
    ml_models = {}
    for target in target_cols:
        try:
            # Try to load individual models first
            model_path = f"models/regressors/{target}_model.pkl"
            ml_models[target] = joblib.load(model_path)
            print(f"✅ Loaded model for: {target}")
        except:
            # Fallback to single model
            try:
                ml_models[target] = joblib.load("models/regressors/xgboost_model.pkl")
                print(f"⚠️ Using fallback model for: {target}")
            except:
                print(f"❌ No model found for: {target}")
                return None
    
    # YOLO detection
    results = yolo_model.predict(image_path, imgsz=640)
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    
    # Compute pairwise distances
    pixel_distances = []
    for i, j in combinations(range(len(keypoints)), 2):
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        pixel_distances.append(dist)
    
    pixel_distances = np.array(pixel_distances)
    
    # Convert to cm (using your scaling parameters)
    PIXEL_TO_CM_SLOPE = 0.0501
    PIXEL_TO_CM_INTERCEPT = 34.7894
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    
    # Ensure correct feature dimension
    if len(cm_distances) != len(feature_cols):
        print(f"⚠️ Feature dimension mismatch: {len(cm_distances)} vs {len(feature_cols)}")
        return None
    
    # Make predictions
    measurements = {}
    for target, model in ml_models.items():
        try:
            prediction = model.predict([cm_distances])[0]
            measurements[target] = prediction
        except Exception as e:
            print(f"❌ Prediction error for {target}: {e}")
            measurements[target] = 0.0
    
    print("📏 PREDICTED MEASUREMENTS:")
    for measurement, value in measurements.items():
        print(f"  {measurement}: {value:.1f} cm")
    
    return measurements

# Test the corrected pipeline
print("\n" + "="*50)
print("🚀 TESTING CORRECTED INFERENCE PIPELINE")
print("="*50)

test_images = ["data/images/29.png", "data/images/30.png", "data/images/121.jpg"]
for img_path in test_images:
    try:
        results = corrected_inference_pipeline(img_path)
        if results:
            # Check if predictions are varying
            values = list(results.values())
            if len(set([round(v, 1) for v in values])) == 1:
                print("⚠️ Predictions still constant - need to retrain models")
            else:
                print("✅ Predictions are varying!")
    except Exception as e:
        print(f"❌ Error: {e}")

📊 Loading features data for model validation...
Dataset shape: (276, 19)
Features: 15, Targets: 3

🎯 Target distributions:
  oblique_body_length_cm: min=67.0, max=174.0, mean=139.5
  withers_height_cm: min=93.0, max=165.0, mean=122.6
  girth_length_cm: min=98.0, max=230.0, mean=183.7

🧪 TESTING EXISTING MODEL...
✅ Existing model loaded
📈 Unique predictions: 83
📊 Prediction range: 98.00 to 230.00
✅ Model seems to be working

🚀 TESTING CORRECTED INFERENCE PIPELINE

🔍 Processing: data/images/29.png
⚠️ Using fallback model for: oblique_body_length_cm
⚠️ Using fallback model for: withers_height_cm
⚠️ Using fallback model for: girth_length_cm

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 40.0ms
Speed: 5.4ms preprocess, 40.0ms inference, 3.7ms postprocess per image at shape (1, 3, 480, 640)
📏 PREDICTED MEASUREMENTS:
  oblique_body_length_cm: 180.7 cm
  withers_height_cm: 180.7 cm
  girth_length_cm: 180.7 cm
⚠️ Predictions still constant - need to retrain mod

In [63]:
# ==============================
# STEP 9: FIX FEATURE ORDER MISMATCH
# ==============================
# The problem: itertools.combinations returns pairs in different order
# than how they were created during training!

import pandas as pd
import numpy as np
from itertools import combinations
import joblib
from ultralytics import YOLO

def get_correct_feature_order():
    """Get the exact feature order used during training"""
    df = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in df.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    
    print("📋 TRAINING FEATURE ORDER:")
    feature_pairs = []
    for col in feature_cols:
        i, j = map(int, col.split('_'))
        feature_pairs.append((i, j))
        print(f"  {col}: ({i}, {j})")
    
    return feature_pairs

def compute_distances_correct_order(keypoints, correct_order):
    """Compute distances in EXACTLY the same order as training"""
    distances = []
    for i, j in correct_order:
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        distances.append(dist)
    return np.array(distances)

def fixed_inference_pipeline(image_path):
    """FINALLY FIXED: Uses exact same feature order as training"""
    print(f"\n🔍 Processing: {image_path}")
    
    # Load models
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    ml_model = joblib.load("models/regressors/xgboost_model.pkl")
    feature_scaler = joblib.load("models/regressors/feature_scaler.pkl")
    
    # Get the EXACT feature order from training
    correct_feature_order = get_correct_feature_order()
    
    # YOLO detection
    results = yolo_model.predict(image_path, imgsz=640)
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    print(f"🎯 Detected {len(keypoints)} keypoints")
    
    # Compute distances in CORRECT ORDER
    pixel_distances = compute_distances_correct_order(keypoints, correct_feature_order)
    
    # Convert to cm
    PIXEL_TO_CM_SLOPE = 0.0501
    PIXEL_TO_CM_INTERCEPT = 34.7894
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    
    print(f"📏 CM distances range: {cm_distances.min():.1f} to {cm_distances.max():.1f} cm")
    
    # Scale features
    features_scaled = feature_scaler.transform([cm_distances])
    
    # Make predictions for each target
    measurements = {}
    targets = ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']
    
    # Since we have one model, we'll use it for all targets (temporary)
    # In production, you should train separate models
    for target in targets:
        prediction = ml_model.predict(features_scaled)[0]
        measurements[target] = prediction
    
    print("📊 PREDICTED MEASUREMENTS:")
    for measurement, value in measurements.items():
        print(f"  {measurement}: {value:.1f} cm")
    
    return measurements

# ==============================
# CREATE SEPARATE MODELS FOR EACH TARGET
# ==============================

def create_individual_models():
    """Create separate models for each measurement target"""
    print("\n🔄 CREATING INDIVIDUAL MODELS FOR EACH TARGET...")
    
    df = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in df.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    targets = ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']
    
    from sklearn.model_selection import train_test_split
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_absolute_error, r2_score
    
    models = {}
    
    for target in targets:
        print(f"\n🎯 Training model for: {target}")
        
        X = df[feature_cols]
        y = df[target]
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        model = XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42
        )
        
        model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        models[target] = model
        print(f"  ✅ MAE: {mae:.2f} cm, R²: {r2:.3f}")
        
        # Save model
        joblib.dump(model, f"models/regressors/individual_{target}_model.pkl")
    
    return models

def fixed_individual_inference(image_path):
    """Use individual models with correct feature order"""
    print(f"\n🔍 Processing: {image_path}")
    
    # Load models
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    feature_scaler = joblib.load("models/regressors/feature_scaler.pkl")
    
    # Get correct feature order
    correct_feature_order = get_correct_feature_order()
    
    # Load individual models
    models = {}
    targets = ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']
    for target in targets:
        try:
            models[target] = joblib.load(f"models/regressors/individual_{target}_model.pkl")
            print(f"✅ Loaded individual model for: {target}")
        except:
            print(f"❌ Could not load model for: {target}")
            return None
    
    # YOLO detection
    results = yolo_model.predict(image_path, imgsz=640)
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    
    # Compute distances in CORRECT ORDER
    pixel_distances = compute_distances_correct_order(keypoints, correct_feature_order)
    
    # Convert to cm
    PIXEL_TO_CM_SLOPE = 0.0501
    PIXEL_TO_CM_INTERCEPT = 34.7894
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    
    # Scale features
    features_scaled = feature_scaler.transform([cm_distances])
    
    # Make predictions with individual models
    measurements = {}
    for target, model in models.items():
        prediction = model.predict(features_scaled)[0]
        measurements[target] = prediction
    
    print("📊 PREDICTED MEASUREMENTS:")
    for measurement, value in measurements.items():
        print(f"  {measurement}: {value:.1f} cm")
    
    return measurements

# ==============================
# TEST THE FIXED PIPELINE
# ==============================

print("🚀 TESTING FIXED INFERENCE PIPELINE")
print("=" * 50)

# First test with correct feature order but single model
print("\n1. Testing with correct feature order (single model):")
test_images = ["data/images/29.png", "data/images/30.png", "data/images/121.jpg"]
for img_path in test_images:
    try:
        results = fixed_inference_pipeline(img_path)
        if results:
            values = list(results.values())
            if len(set([round(v, 1) for v in values])) == 1:
                print("   ⚠️ Still constant - need individual models")
            else:
                print("   ✅ Varying predictions!")
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Create individual models
print("\n2. Creating individual models...")
individual_models = create_individual_models()

# Test with individual models
print("\n3. Testing with individual models:")
for img_path in test_images:
    try:
        results = fixed_individual_inference(img_path)
        if results:
            values = list(results.values())
            unique_values = len(set([round(v, 1) for v in values]))
            if unique_values > 1:
                print("   ✅ SUCCESS! Predictions are now varying!")
            else:
                print("   ⚠️ Predictions still similar")
    except Exception as e:
        print(f"   ❌ Error: {e}")

🚀 TESTING FIXED INFERENCE PIPELINE

1. Testing with correct feature order (single model):

🔍 Processing: data/images/29.png
📋 TRAINING FEATURE ORDER:
  0_1: (0, 1)
  0_2: (0, 2)
  0_3: (0, 3)
  0_4: (0, 4)
  0_5: (0, 5)
  1_2: (1, 2)
  1_3: (1, 3)
  1_4: (1, 4)
  1_5: (1, 5)
  2_3: (2, 3)
  2_4: (2, 4)
  2_5: (2, 5)
  3_4: (3, 4)
  3_5: (3, 5)
  4_5: (4, 5)

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 38.9ms
Speed: 4.6ms preprocess, 38.9ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)
🎯 Detected 6 keypoints
📏 CM distances range: 72.2 to 173.2 cm
📊 PREDICTED MEASUREMENTS:
  oblique_body_length_cm: 99.2 cm
  withers_height_cm: 99.2 cm
  girth_length_cm: 99.2 cm
   ⚠️ Still constant - need individual models

🔍 Processing: data/images/30.png
📋 TRAINING FEATURE ORDER:
  0_1: (0, 1)
  0_2: (0, 2)
  0_3: (0, 3)
  0_4: (0, 4)
  0_5: (0, 5)
  1_2: (1, 2)
  1_3: (1, 3)
  1_4: (1, 4)
  1_5: (1, 5)
  2_3: (2, 3)
  2_4: (2, 4)
  2_5: (2, 5)
  3

In [65]:
# ==============================
# STEP 10: FINAL COMPLETE FIX - PROPER FEATURE SCALING
# ==============================
# The problem: We're using the wrong scaler or the features aren't being scaled properly

import pandas as pd
import numpy as np
import joblib
from ultralytics import YOLO
from sklearn.preprocessing import StandardScaler

def debug_feature_scaling():
    """Debug the feature scaling issue"""
    print("🔍 DEBUGGING FEATURE SCALING...")
    
    # Load training data to understand the scaling
    df_train = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in df_train.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    
    print("📊 TRAINING DATA STATISTICS:")
    for col in feature_cols[:3]:  # Show first 3 features
        mean_val = df_train[col].mean()
        std_val = df_train[col].std()
        print(f"  {col}: mean={mean_val:.2f}, std={std_val:.2f}")
    
    # Check the current scaler
    try:
        current_scaler = joblib.load("models/regressors/feature_scaler.pkl")
        print(f"\n📐 CURRENT SCALER: {type(current_scaler).__name__}")
        if hasattr(current_scaler, 'mean_'):
            print(f"  Means: {current_scaler.mean_[:3]}...")
            print(f"  Scales: {current_scaler.scale_[:3]}...")
    except Exception as e:
        print(f"❌ Error loading scaler: {e}")

def final_fixed_inference(image_path):
    """FINAL FIXED VERSION with proper feature handling"""
    print(f"\n🔍 Processing: {image_path}")
    
    # Load models
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    
    # Load individual models
    models = {}
    targets = ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']
    for target in targets:
        models[target] = joblib.load(f"models/regressors/individual_{target}_model.pkl")
    
    # Get correct feature order from training data
    df_train = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in df_train.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    
    # Extract the exact pairs order from feature names
    correct_feature_order = []
    for col in feature_cols:
        i, j = map(int, col.split('_'))
        correct_feature_order.append((i, j))
    
    # YOLO detection
    results = yolo_model.predict(image_path, imgsz=640)
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    print(f"🎯 Detected {len(keypoints)} keypoints")
    
    # Compute distances in CORRECT ORDER
    pixel_distances = []
    for i, j in correct_feature_order:
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        pixel_distances.append(dist)
    
    pixel_distances = np.array(pixel_distances)
    
    # Convert to cm
    PIXEL_TO_CM_SLOPE = 0.0501
    PIXEL_TO_CM_INTERCEPT = 34.7894
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    
    print(f"📏 Raw CM distances - Min: {cm_distances.min():.1f}, Max: {cm_distances.max():.1f}, Mean: {cm_distances.mean():.1f}")
    
    # CRITICAL FIX: Create new scaler or use proper scaling
    # Since the existing scaler might be corrupted, let's create a new one
    X_train = df_train[feature_cols].values
    
    # Create and fit a new scaler (same as during training)
    new_scaler = StandardScaler()
    new_scaler.fit(X_train)
    
    # Scale the features
    features_scaled = new_scaler.transform([cm_distances])
    
    print(f"🔧 Scaled features - Min: {features_scaled.min():.3f}, Max: {features_scaled.max():.3f}, Mean: {features_scaled.mean():.3f}")
    
    # Make predictions
    measurements = {}
    for target, model in models.items():
        prediction = model.predict(features_scaled)[0]
        measurements[target] = prediction
    
    return measurements, cm_distances, features_scaled

def compare_multiple_images():
    """Compare features and predictions across multiple images"""
    print("🔄 COMPARING MULTIPLE IMAGES...")
    
    test_images = ["data/images/29.png", "data/images/30.png", "data/images/121.jpg"]
    
    all_results = []
    
    for img_path in test_images:
        print(f"\n{'='*50}")
        print(f"IMAGE: {img_path}")
        print('='*50)
        
        try:
            measurements, cm_features, scaled_features = final_fixed_inference(img_path)
            
            print("📊 PREDICTED MEASUREMENTS:")
            for measurement, value in measurements.items():
                print(f"  {measurement}: {value:.1f} cm")
            
            # Store results for comparison
            result = {
                'image': img_path,
                'cm_features_mean': cm_features.mean(),
                'cm_features_std': cm_features.std(),
                'scaled_features_mean': scaled_features.mean(),
                'scaled_features_std': scaled_features.std(),
                **measurements
            }
            all_results.append(result)
            
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # Compare results
    if all_results:
        print(f"\n{'='*60}")
        print("📈 COMPARISON ACROSS IMAGES")
        print('='*60)
        
        for result in all_results:
            print(f"\n📷 {result['image']}:")
            print(f"  CM Features - Mean: {result['cm_features_mean']:.1f}, Std: {result['cm_features_std']:.1f}")
            print(f"  Scaled Features - Mean: {result['scaled_features_mean']:.3f}, Std: {result['scaled_features_std']:.3f}")
            print(f"  Oblique: {result['oblique_body_length_cm']:.1f} cm")
            print(f"  Withers: {result['withers_height_cm']:.1f} cm") 
            print(f"  Girth: {result['girth_length_cm']:.1f} cm")
        
        # Check if predictions are actually varying
        oblique_values = [r['oblique_body_length_cm'] for r in all_results]
        withers_values = [r['withers_height_cm'] for r in all_results]
        girth_values = [r['girth_length_cm'] for r in all_results]
        
        print(f"\n🎯 PREDICTION VARIATION:")
        print(f"  Oblique: {np.std(oblique_values):.2f} cm std dev")
        print(f"  Withers: {np.std(withers_values):.2f} cm std dev")
        print(f"  Girth: {np.std(girth_values):.2f} cm std dev")
        
        if (np.std(oblique_values) < 1.0 and np.std(withers_values) < 1.0 and np.std(girth_values) < 1.0):
            print("\n🚨 ALERT: Predictions still too similar across images!")
            print("   This suggests the models are over-regularized or the features aren't discriminative enough")

# ==============================
# NUCLEAR OPTION: RETRAIN EVERYTHING
# ==============================

def nuclear_retrain():
    """Complete retraining from scratch"""
    print("\n💥 NUCLEAR OPTION: RETRAINING EVERYTHING FROM SCRATCH...")
    
    # Load data
    df = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in df.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    targets = ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']
    
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_absolute_error, r2_score
    
    # Create new scaler
    X = df[feature_cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Save the new scaler
    joblib.dump(scaler, "models/regressors/new_feature_scaler.pkl")
    print("✅ Created and saved new feature scaler")
    
    # Train new models
    new_models = {}
    
    for target in targets:
        print(f"\n🎯 Training new model for: {target}")
        
        y = df[target]
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y, test_size=0.2, random_state=42
        )
        
        # Use simpler model with less regularization
        model = XGBRegressor(
            n_estimators=50,  # Fewer trees
            max_depth=4,      # Shallower trees
            learning_rate=0.2, # Higher learning rate
            random_state=42,
            reg_alpha=0,      # No L1 regularization
            reg_lambda=0      # No L2 regularization
        )
        
        model.fit(X_train, y_train)
        
        # Evaluate
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        new_models[target] = model
        print(f"  ✅ MAE: {mae:.2f} cm, R²: {r2:.3f}")
        
        # Save model
        joblib.dump(model, f"models/regressors/new_{target}_model.pkl")
    
    return new_models, scaler

# ==============================
# RUN THE FINAL FIX
# ==============================

print("🚀 RUNNING FINAL COMPLETE FIX")
print("=" * 60)

# First, debug the current state
debug_feature_scaling()

# Test the fixed inference
print("\n1. Testing fixed inference with proper scaling:")
compare_multiple_images()

# If still not working, use nuclear option
print("\n2. Nuclear option: Complete retraining...")
new_models, new_scaler = nuclear_retrain()

print("\n3. Testing with newly trained models...")
# You would need to modify final_fixed_inference to use the new models and scaler
# This should finally give you varying predictions across images!

🚀 RUNNING FINAL COMPLETE FIX
🔍 DEBUGGING FEATURE SCALING...
📊 TRAINING DATA STATISTICS:
  0_1: mean=139.48, std=14.23
  0_2: mean=78.82, std=7.46
  0_3: mean=87.93, std=5.88

📐 CURRENT SCALER: StandardScaler
  Means: [     139.24      78.711      88.011]...
  Scales: [     14.039      7.2549      5.6161]...

1. Testing fixed inference with proper scaling:
🔄 COMPARING MULTIPLE IMAGES...

IMAGE: data/images/29.png

🔍 Processing: data/images/29.png

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 38.9ms
Speed: 5.1ms preprocess, 38.9ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)
🎯 Detected 6 keypoints
📏 Raw CM distances - Min: 72.2, Max: 173.2, Mean: 112.7
🔧 Scaled features - Min: 0.464, Max: 1.845, Mean: 1.203
📊 PREDICTED MEASUREMENTS:
  oblique_body_length_cm: 68.9 cm
  withers_height_cm: 162.3 cm
  girth_length_cm: 100.8 cm

IMAGE: data/images/30.png

🔍 Processing: data/images/30.png

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\

In [67]:
# ==============================
# STEP 10: FINAL COMPLETE FIX - PROPER FEATURE SCALING
# ==============================
# The problem: We're using the wrong scaler or the features aren't being scaled properly

import pandas as pd
import numpy as np
import joblib
from ultralytics import YOLO
from sklearn.preprocessing import StandardScaler

def debug_feature_scaling():
    """Debug the feature scaling issue"""
    print("🔍 DEBUGGING FEATURE SCALING...")
    
    # Load training data to understand the scaling
    df_train = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in df_train.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    
    print("📊 TRAINING DATA STATISTICS:")
    for col in feature_cols[:3]:  # Show first 3 features
        mean_val = df_train[col].mean()
        std_val = df_train[col].std()
        print(f"  {col}: mean={mean_val:.2f}, std={std_val:.2f}")
    
    # Check the current scaler
    try:
        current_scaler = joblib.load("models/regressors/feature_scaler.pkl")
        print(f"\n📐 CURRENT SCALER: {type(current_scaler).__name__}")
        if hasattr(current_scaler, 'mean_'):
            print(f"  Means: {current_scaler.mean_[:3]}...")
            print(f"  Scales: {current_scaler.scale_[:3]}...")
    except Exception as e:
        print(f"❌ Error loading scaler: {e}")

def final_fixed_inference(image_path):
    """FINAL FIXED VERSION with proper feature handling"""
    print(f"\n🔍 Processing: {image_path}")
    
    # Load models
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    
    # Load individual models
    models = {}
    targets = ['oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']
    for target in targets:
        models[target] = joblib.load(f"models/regressors/individual_{target}_model.pkl")
    
    # Get correct feature order from training data
    df_train = pd.read_csv("results/cow_features.csv")
    feature_cols = [col for col in df_train.columns if col not in ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']]
    
    # Extract the exact pairs order from feature names
    correct_feature_order = []
    for col in feature_cols:
        i, j = map(int, col.split('_'))
        correct_feature_order.append((i, j))
    
    # YOLO detection
    results = yolo_model.predict(image_path, imgsz=640)
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    print(f"🎯 Detected {len(keypoints)} keypoints")
    
    # Compute distances in CORRECT ORDER
    pixel_distances = []
    for i, j in correct_feature_order:
        dist = np.linalg.norm(keypoints[i] - keypoints[j])
        pixel_distances.append(dist)
    
    pixel_distances = np.array(pixel_distances)
    
    # Convert to cm
    PIXEL_TO_CM_SLOPE = 0.0501
    PIXEL_TO_CM_INTERCEPT = 34.7894
    cm_distances = pixel_distances * PIXEL_TO_CM_SLOPE + PIXEL_TO_CM_INTERCEPT
    
    print(f"📏 Raw CM distances - Min: {cm_distances.min():.1f}, Max: {cm_distances.max():.1f}, Mean: {cm_distances.mean():.1f}")
    
    # CRITICAL FIX: Create new scaler or use proper scaling
    # Since the existing scaler might be corrupted, let's create a new one
    X_train = df_train[feature_cols].values
    
    # Create and fit a new scaler (same as during training)
    new_scaler = StandardScaler()
    new_scaler.fit(X_train)
    
    # Scale the features
    features_scaled = new_scaler.transform([cm_distances])
    
    print(f"🔧 Scaled features - Min: {features_scaled.min():.3f}, Max: {features_scaled.max():.3f}, Mean: {features_scaled.mean():.3f}")
    
    # Make predictions
    measurements = {}
    for target, model in models.items():
        prediction = model.predict(features_scaled)[0]
        measurements[target] = prediction
    
    return measurements, cm_distances, features_scaled

def compare_multiple_images():
    """Compare features and predictions across multiple images"""
    print("🔄 COMPARING MULTIPLE IMAGES...")
    
    test_images = ["data/images/29.png", "data/images/30.png", "data/images/121.jpg"]
    
    all_results = []
    
    for img_path in test_images:
        print(f"\n{'='*50}")
        print(f"IMAGE: {img_path}")
        print('='*50)
        
        try:
            measurements, cm_features, scaled_features = final_fixed_inference(img_path)
            
            print("📊 PREDICTED MEASUREMENTS:")
            for measurement, value in measurements.items():
                print(f"  {measurement}: {value:.1f} cm")
            
            # Store results for comparison
            result = {
                'image': img_path,
                'cm_features_mean': cm_features.mean(),
                'cm_features_std': cm_features.std(),
                'scaled_features_mean': scaled_features.mean(),
                'scaled_features_std': scaled_features.std(),
                **measurements
            }
            all_results.append(result)
            
        except Exception as e:
            print(f"❌ Error: {e}")
    
    # Compare results
    if all_results:
        print(f"\n{'='*60}")
        print("📈 COMPARISON ACROSS IMAGES")
        print('='*60)
        
        for result in all_results:
            print(f"\n📷 {result['image']}:")
            print(f"  CM Features - Mean: {result['cm_features_mean']:.1f}, Std: {result['cm_features_std']:.1f}")
            print(f"  Scaled Features - Mean: {result['scaled_features_mean']:.3f}, Std: {result['scaled_features_std']:.3f}")
            print(f"  Oblique: {result['oblique_body_length_cm']:.1f} cm")
            print(f"  Withers: {result['withers_height_cm']:.1f} cm") 
            print(f"  Girth: {result['girth_length_cm']:.1f} cm")
        
        # Check if predictions are actually varying
        oblique_values = [r['oblique_body_length_cm'] for r in all_results]
        withers_values = [r['withers_height_cm'] for r in all_results]
        girth_values = [r['girth_length_cm'] for r in all_results]
        
        print(f"\n🎯 PREDICTION VARIATION:")
        print(f"  Oblique: {np.std(oblique_values):.2f} cm std dev")
        print(f"  Withers: {np.std(withers_values):.2f} cm std dev")
        print(f"  Girth: {np.std(girth_values):.2f} cm std dev")
        
        if (np.std(oblique_values) < 1.0 and np.std(withers_values) < 1.0 and np.std(girth_values) < 1.0):
            print("\n🚨 ALERT: Predictions still too similar across images!")
            print("   This suggests the models are over-regularized or the features aren't discriminative enough")


print("🚀 RUNNING FINAL COMPLETE FIX")
print("=" * 60)

# First, debug the current state
debug_feature_scaling()

# Test the fixed inference
print("\n1. Testing fixed inference with proper scaling:")
compare_multiple_images()

🚀 RUNNING FINAL COMPLETE FIX
🔍 DEBUGGING FEATURE SCALING...
📊 TRAINING DATA STATISTICS:
  0_1: mean=139.48, std=14.23
  0_2: mean=78.82, std=7.46
  0_3: mean=87.93, std=5.88

📐 CURRENT SCALER: StandardScaler
  Means: [     139.24      78.711      88.011]...
  Scales: [     14.039      7.2549      5.6161]...

1. Testing fixed inference with proper scaling:
🔄 COMPARING MULTIPLE IMAGES...

IMAGE: data/images/29.png

🔍 Processing: data/images/29.png

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 39.1ms
Speed: 5.4ms preprocess, 39.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
🎯 Detected 6 keypoints
📏 Raw CM distances - Min: 72.2, Max: 173.2, Mean: 112.7
🔧 Scaled features - Min: 0.464, Max: 1.845, Mean: 1.203
📊 PREDICTED MEASUREMENTS:
  oblique_body_length_cm: 68.9 cm
  withers_height_cm: 162.3 cm
  girth_length_cm: 100.8 cm

IMAGE: data/images/30.png

🔍 Processing: data/images/30.png

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\

In [70]:
# ==============================
# STEP 1: DIRECT MEASUREMENT CALCULATION (FIXED)
# ==============================
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

def calculate_scaling_factors_from_excel():
    """Calculate cm-per-pixel ratios from Excel training data"""
    
    # Load your Excel measurements
    df = pd.read_excel("data/measurements.xlsx")
    df['image_id'] = df['image_id'].astype(str)
    
    scaling_data = []
    
    print(f"Processing {len(df)} images from Excel...")
    
    for idx, row in df.iterrows():
        image_id = row['image_id']
        
        # Load corresponding keypoints
        keypoints = load_keypoints_from_yolo(f"data/labels/{image_id}.txt", image_id)
        if keypoints is None:
            print(f"  ⚠️ Skipping {image_id} - keypoints not found")
            continue
            
        # Calculate pixel distances for the 3 reference pairs
        oblique_pixels = np.linalg.norm(keypoints[0] - keypoints[1])
        withers_pixels = np.linalg.norm(keypoints[2] - keypoints[3])
        girth_pixels = np.linalg.norm(keypoints[4] - keypoints[5])
        
        # Get actual measurements from Excel
        oblique_cm = row['oblique_body_length_cm']
        withers_cm = row['withers_height_cm']
        girth_cm = row['girth_length_cm']
        
        # Calculate scaling factors
        oblique_ratio = oblique_cm / oblique_pixels if oblique_pixels > 0 else 0
        withers_ratio = withers_cm / withers_pixels if withers_pixels > 0 else 0
        girth_ratio = girth_cm / girth_pixels if girth_pixels > 0 else 0
        
        scaling_data.append({
            'image_id': image_id,
            'oblique_ratio': oblique_ratio,
            'withers_ratio': withers_ratio,
            'girth_ratio': girth_ratio,
            'oblique_pixels': oblique_pixels,
            'withers_pixels': withers_pixels,
            'girth_pixels': girth_pixels
        })
        
        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{len(df)} images")
    
    if not scaling_data:
        print("❌ No valid scaling data collected! Check file paths.")
        return 0.1, 0.1, 0.1  # Default fallback values
    
    scaling_df = pd.DataFrame(scaling_data)
    
    # Calculate average scaling factors
    avg_oblique_ratio = scaling_df['oblique_ratio'].mean()
    avg_withers_ratio = scaling_df['withers_ratio'].mean()
    avg_girth_ratio = scaling_df['girth_ratio'].mean()
    
    print("\n📏 AVERAGE SCALING FACTORS (cm/pixel):")
    print(f"  Oblique body length: {avg_oblique_ratio:.6f}")
    print(f"  Withers height: {avg_withers_ratio:.6f}")
    print(f"  Girth length: {avg_girth_ratio:.6f}")
    
    # Show some statistics
    print(f"\n📊 Based on {len(scaling_data)} valid images")
    print(f"  Oblique ratios range: {scaling_df['oblique_ratio'].min():.6f} to {scaling_df['oblique_ratio'].max():.6f}")
    print(f"  Withers ratios range: {scaling_df['withers_ratio'].min():.6f} to {scaling_df['withers_ratio'].max():.6f}")
    print(f"  Girth ratios range: {scaling_df['girth_ratio'].min():.6f} to {scaling_df['girth_ratio'].max():.6f}")
    
    return avg_oblique_ratio, avg_withers_ratio, avg_girth_ratio

def load_keypoints_from_yolo(label_path, image_id):
    """Load keypoints from YOLO format label file"""
    try:
        # Check if label file exists
        if not os.path.exists(label_path):
            # Try alternative path
            alt_path = f"data/labels_yolo/{image_id}.txt"
            if os.path.exists(alt_path):
                label_path = alt_path
            else:
                return None
        
        # Find image to get dimensions
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
            candidate = f"data/images/{image_id}{ext}"
            if os.path.exists(candidate):
                img_path = candidate
                break
        
        if not img_path:
            # Try side_view folder
            for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
                candidate = f"data/images/{image_id}{ext}"
                if os.path.exists(candidate):
                    img_path = candidate
                    break
        
        if not img_path:
            return None
            
        img = cv2.imread(img_path)
        if img is None:
            return None
            
        h, w = img.shape[:2]
        
        # Read YOLO label
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        if not lines:
            return None
            
        # Parse first object's keypoints
        parts = list(map(float, lines[0].strip().split()))
        if len(parts) < 5 + 6 * 2:  # class + bbox + 6 keypoints
            return None
            
        # Extract keypoints (skip class_id and bbox: first 5 values)
        kpt_data = parts[5:5 + 6 * 2]
        keypoints = []
        
        for i in range(6):
            x = kpt_data[i * 2] * w
            y = kpt_data[i * 2 + 1] * h
            keypoints.append([x, y])
        
        return np.array(keypoints)
        
    except Exception as e:
        print(f"Error loading keypoints for {image_id}: {e}")
        return None

def direct_measurement_pipeline(image_path, oblique_ratio, withers_ratio, girth_ratio):
    """Direct measurement calculation for new images"""
    print(f"\n🔍 Processing: {image_path}")
    
    # Load YOLO model
    yolo_model = YOLO("runs/pose/yolov8_cow_pose_v1/weights/best.pt")
    
    # Detect keypoints
    results = yolo_model.predict(image_path, imgsz=640)
    
    if len(results) == 0 or len(results[0].keypoints.xy) == 0:
        print("❌ No keypoints detected")
        return None
    
    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    print(f"✅ Detected {len(keypoints)} keypoints")
    
    # Calculate pixel distances for the 3 reference pairs
    oblique_pixels = np.linalg.norm(keypoints[0] - keypoints[1])
    withers_pixels = np.linalg.norm(keypoints[2] - keypoints[3])
    girth_pixels = np.linalg.norm(keypoints[4] - keypoints[5])
    
    print(f"📐 Pixel distances:")
    print(f"  Oblique (0-1): {oblique_pixels:.1f} pixels")
    print(f"  Withers (2-3): {withers_pixels:.1f} pixels") 
    print(f"  Girth (4-5): {girth_pixels:.1f} pixels")
    
    # Convert to cm using scaling factors
    oblique_cm = oblique_pixels * oblique_ratio
    withers_cm = withers_pixels * withers_ratio
    girth_cm = girth_pixels * girth_ratio
    
    measurements = {
        'oblique_body_length_cm': oblique_cm,
        'withers_height_cm': withers_cm,
        'girth_length_cm': girth_cm
    }
    
    print("📏 CALCULATED MEASUREMENTS:")
    print(f"  Oblique body length: {oblique_cm:.1f} cm")
    print(f"  Withers height: {withers_cm:.1f} cm")
    print(f"  Girth length: {girth_cm:.1f} cm")
    
    # Create visualization
    img = cv2.imread(image_path)
    vis_img = create_measurement_visualization(img, keypoints, measurements)
    output_path = image_path.replace('.jpg', '_direct_measurements.jpg').replace('.png', '_direct_measurements.jpg')
    cv2.imwrite(output_path, vis_img)
    print(f"✅ Visualization saved: {output_path}")
    
    return measurements

def create_measurement_visualization(image, keypoints, measurements):
    """Create visualization with keypoints and measurements"""
    img_vis = image.copy()
    
    # Define skeleton pairs and colors
    skeleton_pairs = [(0, 1), (2, 3), (4, 5)]
    colors = [(0, 255, 0), (255, 0, 0), (0, 0, 255)]  # Green, Blue, Red
    
    # Draw skeleton lines
    for (i, j), color in zip(skeleton_pairs, colors):
        if i < len(keypoints) and j < len(keypoints):
            pt1 = (int(keypoints[i][0]), int(keypoints[i][1]))
            pt2 = (int(keypoints[j][0]), int(keypoints[j][1]))
            cv2.line(img_vis, pt1, pt2, color, 4)
    
    # Draw keypoints
    for idx, (x, y) in enumerate(keypoints):
        center = (int(x), int(y))
        cv2.circle(img_vis, center, 6, (0, 0, 255), -1)
        cv2.putText(img_vis, str(idx), (center[0] + 10, center[1]), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    
    # Add measurements text
    y_offset = 40
    measurement_names = {
        'oblique_body_length_cm': 'Oblique Length',
        'withers_height_cm': 'Withers Height', 
        'girth_length_cm': 'Girth Length'
    }
    
    for measurement, display_name in measurement_names.items():
        value = measurements[measurement]
        text = f"{display_name}: {value:.1f} cm"
        cv2.putText(img_vis, text, (20, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 3)
        cv2.putText(img_vis, text, (20, y_offset), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        y_offset += 35
    
    return img_vis

# ==============================
# MAIN EXECUTION
# ==============================

print("🐄 DIRECT MEASUREMENT CALCULATION SYSTEM")
print("=" * 50)

# Step 1: Calculate scaling factors from training data
print("1. Calculating scaling factors from Excel data...")
oblique_ratio, withers_ratio, girth_ratio = calculate_scaling_factors_from_excel()

# Step 2: Test on new images
print("\n2. Testing on new images...")
test_images = ["data/images/29.png", "data/images/30.png", "data/images/121.jpg"]

for img_path in test_images:
    try:
        if os.path.exists(img_path):
            measurements = direct_measurement_pipeline(img_path, oblique_ratio, withers_ratio, girth_ratio)
            if measurements:
                print(f"✅ Successfully processed {img_path}")
        else:
            print(f"❌ Image not found: {img_path}")
    except Exception as e:
        print(f"❌ Error processing {img_path}: {e}")

print("\n🎯 SYSTEM READY!")
print(f"Use these scaling factors for new images:")
print(f"  Oblique ratio: {oblique_ratio:.6f} cm/pixel")
print(f"  Withers ratio: {withers_ratio:.6f} cm/pixel") 
print(f"  Girth ratio: {girth_ratio:.6f} cm/pixel")

🐄 DIRECT MEASUREMENT CALCULATION SYSTEM
1. Calculating scaling factors from Excel data...
Processing 138 images from Excel...
  Processed 10/138 images
  Processed 20/138 images
  Processed 30/138 images
  Processed 40/138 images
  Processed 50/138 images
  Processed 60/138 images
  Processed 70/138 images
  Processed 80/138 images
  Processed 90/138 images
  Processed 100/138 images
  Processed 110/138 images
  Processed 120/138 images
  Processed 130/138 images
  ⚠️ Skipping 132 - keypoints not found

📏 AVERAGE SCALING FACTORS (cm/pixel):
  Oblique body length: 0.067016
  Withers height: 0.067705
  Girth length: 0.189659

📊 Based on 137 valid images
  Oblique ratios range: 0.038090 to 0.086222
  Withers ratios range: 0.048964 to 0.098688
  Girth ratios range: 0.124826 to 0.249915

2. Testing on new images...

🔍 Processing: data/images/29.png

image 1/1 d:\Projects\VETRA\Codes\VETRA v1.1\data\images\29.png: 480x640 1 cow, 39.4ms
Speed: 6.5ms preprocess, 39.4ms inference, 4.3ms postpro